In [0]:
silver_catalog_path='pricing_data.silver.daily_pricing'

In [0]:
from pyspark.sql.functions import col, coalesce, current_date, to_date, row_number, current_timestamp
from pyspark.sql.types import LongType, IntegerType, DecimalType
from pyspark.sql.window import Window

In [0]:
bronze_df=spark.sql("""
select * from pricing_data.bronze.daily_pricing where lakehouse_updated_date>
(select coalesce(max(Process_datetime), date('1900-01-01')) from pricing_data.process_run_logs.pipeline_process_control where Process_file_name='pricing_data_transformation_silver' and Process_status='Completed')
""")

In [0]:
selected_df=bronze_df.select(
    coalesce(to_date(col('DATE_OF_PRICING'), 'MM/dd/yyyy'), current_date()).alias('DATE_OF_PRICING'),
    col('ROW_ID').cast(LongType()).alias('ROW_ID'),
    col('STATE_NAME'),
    col('MARKET_NAME'),
    col('PRODUCTGROUP_NAME'),
    col('PRODUCT_NAME'),
    col('VARIETY'),
    col('ARRIVAL_IN_TONNES').try_cast(DecimalType(12, 2)).alias('ARRIVAL_IN_TONNES'),
    col('MINIMUM_PRICE').try_cast(DecimalType(18, 2)).alias('MINIMUM_PRICE'),
    col('MAXIMUM_PRICE').try_cast(DecimalType(18, 2)).alias('MAXIMUM_PRICE'),
    col('MODAL_PRICE').cast(DecimalType(18, 2)).alias('MODAL_PRICE'),
    col('lakehouse_updated_date').alias('bronze_file_updated_date')
)

In [0]:
selected_df.createOrReplaceTempView('selected_data')

In [0]:

clean_df=selected_df.withColumn("rn", row_number().over(Window.partitionBy('DATE_OF_PRICING', 'ROW_ID').orderBy(col('bronze_file_updated_date').desc()))).filter(col("rn")==1).drop("rn", "bronze_file_updated_date")

valid_df=clean_df.filter((col('DATE_OF_PRICING').isNotNull()) & (col('ROW_ID').isNotNull()) & (col('MODAL_PRICE')>0))

quarntine_df=clean_df.filter((col('DATE_OF_PRICING').isNull()) | (col('ROW_ID').isNull()) | (col('MODAL_PRICE')<=0))

valid_df=valid_df.fillna('NA', subset=['STATE_NAME', 'MARKET_NAME', 'PRODUCTGROUP_NAME','PRODUCT_NAME', 'VARIETY'])

valid_df=valid_df.fillna(0.0, subset=['ARRIVAL_IN_TONNES'])

valid_df=valid_df.fillna(0, subset=['MINIMUM_PRICE', 'MAXIMUM_PRICE'])

final_df=valid_df.withColumn('lakehouse_inserted_date', current_timestamp()).withColumn('lakehouse_updated_date', current_timestamp())

In [0]:
spark.sql("""insert into pricing_data.process_run_logs.fileprocess_datacount_log
    select 'pricing_data_transformation_silver',
    'daily_pricing',
    count(*),
    sum(case when DATE_OF_PRICING is not null and ROW_ID is not null and MODAL_PRICE>0 then 1 else 0 end) as valid_count,
    sum(case when DATE_OF_PRICING is null or ROW_ID is null or MODAL_PRICE<=0 then 1 else 0 end) as quarntine_count,
    current_timestamp()
    from selected_data
    """)

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists(silver_catalog_path):

    delta_table=DeltaTable.forName(spark, silver_catalog_path)

    delta_table.alias('target').merge(final_df.alias('source'), 'target.DATE_OF_PRICING=source.DATE_OF_PRICING and target.ROW_ID=source.ROW_ID').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    final_df.write.format("delta").mode("overwrite").saveAsTable(silver_catalog_path)